## Data Quality Assessment Overview

In this section, we evaluate the overall quality of the dataset before proceeding to analytical tasks. We begin by normalizing the nested JSON structure into a tabular format suitable for analysis, ensuring that all attributes are accessible at the column level.

We then assess key data quality dimensions, including completeness (missing values), structural consistency, duplication, and data type validity. These checks allow us to identify potential governance gaps, inconsistencies in data collection, and fields that may require cleaning or contextual interpretation before further analysis.

### Data Quality Checklist

The dataset is assessed across the following data quality dimensions:

1. **Completeness** – Evaluation of missing or incomplete records.
2. **Uniqueness** – Detection of duplicate records and duplicate unique identifiers (_id).
3. **Consistency** – Assessment of categorical coding consistency and cross-field logical integrity.
4. **Validity** – Verification of acceptable ranges, formats, and allowed values.
5. **Accuracy (Plausibility)** – Identification of implausible but not necessarily invalid values.

Each identified issue will be quantified (count and percentage) and, where appropriate, remediation steps will be demonstrated in code.

In [73]:
import re,json
import pandas as pd
from pathlib import Path
import numpy as np

In [74]:
path = Path("..") / "Data" / "raw_credit_applications.json"

with open(path, "r", encoding="utf-8") as f:
    data = json.load(f)

df = pd.DataFrame(data)
df.head()

,_id,applicant_info,financials,spending_behavior,decision,processing_timestamp,loan_purpose,notes
0,app_200,"{'full_name': 'Jerry Smith', 'email': 'jerry.s...","{'annual_income': 73000, 'credit_history_month...","[{'category': 'Shopping', 'amount': 480}, {'ca...","{'loan_approved': False, 'rejection_reason': '...",2024-01-15T00:00:00Z,NaN,NaN
1,app_037,"{'full_name': 'Brandon Walker', 'email': 'bran...","{'annual_income': 78000, 'credit_history_month...","[{'category': 'Rent', 'amount': 608}, {'catego...","{'loan_approved': False, 'rejection_reason': '...",NaN,NaN,NaN
2,app_215,"{'full_name': 'Scott Moore', 'email': 'scott.m...","{'annual_income': 61000, 'credit_history_month...","[{'category': 'Rent', 'amount': 109}]","{'loan_approved': True, 'interest_rate': 3.7, ...",NaN,vacation,NaN
3,app_024,"{'full_name': 'Thomas Lee', 'email': 'thomas.l...","{'annual_income': 103000, 'credit_history_mont...","[{'category': 'Fitness', 'amount': 575}]","{'loan_approved': True, 'interest_rate': 4.3, ...",NaN,NaN,NaN
4,app_184,"{'full_name': 'Brian Rodriguez', 'email': 'bri...","{'annual_income': 57000, 'credit_history_month...","[{'category': 'Entertainment', 'amount': 463}]","{'loan_approved': False, 'rejection_reason': '...",2024-01-15T00:00:00Z,NaN,NaN


The raw JSON dataset was successfully loaded and converted into a Pandas DataFrame. The structure confirms the presence of nested fields (e.g., applicant information, financial data, and decision details), which will require normalization before performing data quality analysis.

In [75]:
df.columns

Index(['_id', 'applicant_info', 'financials', 'spending_behavior', 'decision',
       'processing_timestamp', 'loan_purpose', 'notes'],
      dtype='object')

In [76]:
flat_raw = pd.json_normalize(data, sep=".")
flat = flat_raw.copy()
flat.head()

,_id,spending_behavior,processing_timestamp,applicant_info.full_name,applicant_info.email,applicant_info.ssn,applicant_info.ip_address,applicant_info.gender,applicant_info.date_of_birth,applicant_info.zip_code,...,financials.credit_history_months,financials.debt_to_income,financials.savings_balance,decision.loan_approved,decision.rejection_reason,loan_purpose,decision.interest_rate,decision.approved_amount,financials.annual_salary,notes
0,app_200,"[{'category': 'Shopping', 'amount': 480}, {'ca...",2024-01-15T00:00:00Z,Jerry Smith,jerry.smith17@hotmail.com,596-64-4340,192.168.48.155,Male,2001-03-09,10036,...,23,0.20,31212,False,algorithm_risk_score,NaN,NaN,NaN,NaN,NaN
1,app_037,"[{'category': 'Rent', 'amount': 608}, {'catego...",NaN,Brandon Walker,brandon.walker2@yahoo.com,425-69-4784,10.1.102.112,M,1992-03-31,10032,...,51,0.18,17915,False,algorithm_risk_score,NaN,NaN,NaN,NaN,NaN
2,app_215,"[{'category': 'Rent', 'amount': 109}]",NaN,Scott Moore,scott.moore94@mail.com,370-78-5178,10.240.193.250,Male,1989-10-24,10075,...,41,0.21,37909,True,NaN,vacation,3.7,59000.0,NaN,NaN
3,app_024,"[{'category': 'Fitness', 'amount': 575}]",NaN,Thomas Lee,thomas.lee6@protonmail.com,194-35-1833,192.168.175.67,Male,1983-04-25,10077,...,70,0.35,0,True,NaN,NaN,4.3,34000.0,NaN,NaN
4,app_184,"[{'category': 'Entertainment', 'amount': 463}]",2024-01-15T00:00:00Z,Brian Rodriguez,brian.rodriguez86@aol.com,480-41-2475,172.29.125.105,M,1999-05-21,10080,...,14,0.23,31763,False,algorithm_risk_score,NaN,NaN,NaN,NaN,NaN


In [77]:
flat["spending_behavior"].head()

0    [{'category': 'Shopping', 'amount': 480}, {'ca...
1    [{'category': 'Rent', 'amount': 608}, {'catego...
2                [{'category': 'Rent', 'amount': 109}]
3             [{'category': 'Fitness', 'amount': 575}]
4       [{'category': 'Entertainment', 'amount': 463}]
Name: spending_behavior, dtype: object

In [78]:
# Total rows
n_rows = flat.shape[0]

# Build missing summary table
missing_summary = pd.DataFrame({
    "values_present": flat.notna().sum(),
    "values_missing": flat.isna().sum(),
})

missing_summary["percent_missing"] = (
    missing_summary["values_missing"] / n_rows * 100
).round(2)

# Sort by most missing
missing_summary = missing_summary.sort_values(
    by="percent_missing", ascending=False
)

missing_summary

,values_present,values_missing,percent_missing
notes,2,500,99.60
financials.annual_salary,5,497,99.00
loan_purpose,50,452,90.04
processing_timestamp,62,440,87.65
decision.rejection_reason,210,292,58.17
decision.approved_amount,292,210,41.83
decision.interest_rate,292,210,41.83
financials.annual_income,497,5,1.00
applicant_info.ip_address,497,5,1.00
applicant_info.ssn,497,5,1.00


### Missing Values Analysis

The dataset contains 502 records and 21 variables. The missingness analysis reveals substantial gaps in several fields, particularly `notes` (99.6% missing), `financials.annual_salary` (99.0% missing), `loan_purpose` (90.0% missing), and `processing_timestamp` (87.6% missing). These fields appear either optional, inconsistently recorded, or poorly governed within the data collection process.

Decision-related variables such as `decision.rejection_reason` (58.2% missing) and `decision.approved_amount` / `decision.interest_rate` (41.8% missing) likely reflect structural missingness (e.g., rejection reason only exists for rejected loans). In contrast, core financial and applicant identification fields (e.g., income, SSN, email, credit history, debt-to-income ratio) show near-complete coverage, indicating stronger governance and validation controls for critical underwriting variables.

In [79]:
flat.dtypes

_id                                  object
spending_behavior                    object
processing_timestamp                 object
applicant_info.full_name             object
applicant_info.email                 object
applicant_info.ssn                   object
applicant_info.ip_address            object
applicant_info.gender                object
applicant_info.date_of_birth         object
applicant_info.zip_code              object
financials.annual_income             object
financials.credit_history_months      int64
financials.debt_to_income           float64
financials.savings_balance            int64
decision.loan_approved                 bool
decision.rejection_reason            object
loan_purpose                         object
decision.interest_rate              float64
decision.approved_amount            float64
financials.annual_salary            float64
notes                                object
dtype: object

In [80]:
print("\nDataFrame Information:")
df.info()

print("\nDescriptive Statistics for Numerical Columns:")
print(df.describe())

print("\nDescriptive Statistics for Non-Numerical Columns:")
print(df.describe(include='object'))


DataFrame Information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 502 entries, 0 to 501
Data columns (total 8 columns):
 #   Column                Non-Null Count  Dtype 
---  ------                --------------  ----- 
 0   _id                   502 non-null    object
 1   applicant_info        502 non-null    object
 2   financials            502 non-null    object
 3   spending_behavior     502 non-null    object
 4   decision              502 non-null    object
 5   processing_timestamp  62 non-null     object
 6   loan_purpose          50 non-null     object
 7   notes                 2 non-null      object
dtypes: object(8)
memory usage: 31.5+ KB

Descriptive Statistics for Numerical Columns:
            _id                                     applicant_info  \
count       502                                                502   
unique      500                                                501   
top     app_042  {'full_name': 'Joseph Lopez', 'email': 'joseph...   
freq

## Uniqueness Checks (Duplicate IDs and Duplicate Records)

This section evaluates **uniqueness** at two levels: (1) duplicate application identifiers (`_id`) and (2) fully duplicated rows. Duplicate identifiers indicate potential data integrity issues (e.g., reprocessing of the same application or weak primary key enforcement), while fully duplicated rows suggest redundant ingestion or replication errors. All findings are quantified (count and percentage), and duplicate records are inspected to determine whether they are identical or conflicting.

In [81]:
import pandas as pd
import numpy as np

n = len(flat)

# 1) Duplicate IDs
dup_id_count = flat["_id"].duplicated().sum()
dup_id_pct = round(dup_id_count / n * 100, 2)

print(f"Duplicate _id values: {dup_id_count} ({dup_id_pct}%)")

dup_id_rows = flat[flat["_id"].duplicated(keep=False)].sort_values("_id")
display(dup_id_rows)

# 2) Fully duplicated rows (exclude unhashable columns like lists/dicts)
# Identify columns that contain lists or dicts in at least one row
def is_unhashable_cell(x):
    return isinstance(x, (list, dict))

unhashable_cols = [
    c for c in flat.columns
    if flat[c].apply(is_unhashable_cell).any()
]

hashable_cols = [c for c in flat.columns if c not in unhashable_cols]

print("\nColumns excluded from full-row duplicate check (unhashable types):", unhashable_cols)

dup_row_count = flat[hashable_cols].duplicated().sum()
dup_row_pct = round(dup_row_count / n * 100, 2)

print(f"Fully duplicated rows (excluding {len(unhashable_cols)} unhashable cols): {dup_row_count} ({dup_row_pct}%)")

if dup_row_count > 0:
    display(flat.loc[flat[hashable_cols].duplicated(keep=False)].sort_values("_id"))

# 3) Are duplicate IDs identical or conflicting (based on hashable columns)?
if len(dup_id_rows) > 0:
    def identical_group(g: pd.DataFrame) -> bool:
        return g[hashable_cols].nunique(dropna=False).max() == 1

    conflict_summary = (
        dup_id_rows.groupby("_id")
        .apply(lambda g: pd.Series({
            "rows_with_same_id": len(g),
            "identical_on_hashable_cols": identical_group(g),
            "num_cols_with_differences": int((g[hashable_cols].nunique(dropna=False) > 1).sum())
        }))
        .reset_index()
    )

    print("\nDuplicate _id integrity summary (based on hashable columns):")
    display(conflict_summary)

# 4) Remediation: keep the most complete record per _id
flat["_completeness_score"] = flat.notna().sum(axis=1)

flat_dedup = (
    flat.sort_values(by=["_id", "_completeness_score"], ascending=[True, False])
        .drop_duplicates(subset=["_id"], keep="first")
        .drop(columns=["_completeness_score"])
        .reset_index(drop=True)
)

removed = n - len(flat_dedup)
print(f"\nRemediation applied: kept most complete record per _id.")
print(f"Rows removed: {removed} ({round(removed/n*100, 2)}%)")
print(f"Rows after de-duplication: {flat_dedup.shape[0]}")

# Keep using flat as your working copy (only overwrite if you decide to):
# flat = flat_dedup.copy()

Duplicate _id values: 2 (0.4%)


,_id,spending_behavior,processing_timestamp,applicant_info.full_name,applicant_info.email,applicant_info.ssn,applicant_info.ip_address,applicant_info.gender,applicant_info.date_of_birth,applicant_info.zip_code,...,financials.credit_history_months,financials.debt_to_income,financials.savings_balance,decision.loan_approved,decision.rejection_reason,loan_purpose,decision.interest_rate,decision.approved_amount,financials.annual_salary,notes
383,app_001,"[{'category': 'Fitness', 'amount': 576}]",NaN,Stephanie Nguyen,stephanie.nguyen47@mail.com,427-90-1892,10.121.120.213,Female,1986-05-27,90230,...,37,0.42,0,False,high_dti_ratio,NaN,NaN,NaN,NaN,NaN
455,app_001,"[{'category': 'Fitness', 'amount': 576}]",NaN,Stephanie Nguyen,stephanie.nguyen47@mail.com,NaN,NaN,NaN,NaN,NaN,...,37,0.42,0,False,high_dti_ratio,NaN,NaN,NaN,NaN,DUPLICATE_ENTRY_ERROR
8,app_042,"[{'category': 'Insurance', 'amount': 153}, {'c...",NaN,Joseph Lopez,joseph.lopez1@gmail.com,652-70-5530,192.168.91.142,Male,1990-05-04,10044,...,43,0.41,15974,False,algorithm_risk_score,NaN,NaN,NaN,NaN,NaN
354,app_042,"[{'category': 'Insurance', 'amount': 153}, {'c...",NaN,Joseph Lopez,joseph.lopez1@gmail.com,652-70-5530,192.168.91.142,Male,1990-05-04,10044,...,43,0.41,15974,False,algorithm_risk_score,NaN,NaN,NaN,NaN,RESUBMISSION



Columns excluded from full-row duplicate check (unhashable types): ['spending_behavior']
Fully duplicated rows (excluding 1 unhashable cols): 0 (0.0%)

Duplicate _id integrity summary (based on hashable columns):


C:\Users\roque\AppData\Local\Temp\ipykernel_2628\1450815801.py:44: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: pd.Series({


,_id,rows_with_same_id,identical_on_hashable_cols,num_cols_with_differences
0,app_001,2,False,6
1,app_042,2,False,1



Remediation applied: kept most complete record per _id.
Rows removed: 2 (0.4%)
Rows after de-duplication: 500


### Data Type Standardization and Categorical Consistency

In [82]:
# --- Quick coercions (types) ---
# datetimes

for c in ["processing_timestamp", "applicant_info.date_of_birth"]:
    if c in flat.columns:
        flat[c] = pd.to_datetime(flat[c], errors="coerce")


# numerics (coerce strings -> numbers)
numeric_cols = [
    "financials.annual_income",
    "financials.annual_salary",
    "financials.credit_history_months",
    "financials.debt_to_income",
    "financials.savings_balance",
    "decision.interest_rate",
    "decision.approved_amount",
]
for c in numeric_cols:
    if c in flat.columns:
        flat[c] = pd.to_numeric(flat[c], errors="coerce")

# --- Check coercions ---
print("\nData Types After Coercion:")
print(flat.dtypes)


Data Types After Coercion:
_id                                              object
spending_behavior                                object
processing_timestamp                datetime64[ns, UTC]
applicant_info.full_name                         object
applicant_info.email                             object
applicant_info.ssn                               object
applicant_info.ip_address                        object
applicant_info.gender                            object
applicant_info.date_of_birth             datetime64[ns]
applicant_info.zip_code                          object
financials.annual_income                        float64
financials.credit_history_months                  int64
financials.debt_to_income                       float64
financials.savings_balance                        int64
decision.loan_approved                             bool
decision.rejection_reason                        object
loan_purpose                                     object
decision.interest_ra

In [83]:
# --- Standardize gender (consistency) ---
if "applicant_info.gender" in flat.columns:
    g = flat["applicant_info.gender"].astype(str).str.strip().str.lower()
    gender_map = {
        "m": "male", "male": "male", "man": "male",
        "f": "female", "female": "female", "woman": "female",
        "non-binary": "other", "nonbinary": "other", "other": "other", "o": "other",
        "nan": np.nan, "none": np.nan, "": np.nan
    }
    flat["applicant_info.gender_clean"] = g.map(gender_map).fillna(g.where(g.isin(["male","female","other"]), np.nan))

Key fields were coerced into appropriate data types to ensure analytical validity and structural consistency. Timestamp and date fields were converted to datetime format, while financial variables were coerced to numeric types, allowing detection of invalid or non-convertible entries through automatic `NaN` assignment.

Additionally, gender values were standardized to a controlled vocabulary (`male`, `female`, `other`) to address inconsistent coding (e.g., case differences, abbreviations). This remediation improves categorical consistency and strengthens downstream analytical reliability.